# Duka Electronics: H1 2026 Sales Exploration

## Executive Summary
- Duka Electronics booked **KES 71.8 million** from **1,000 orders** between 1 January and 29 June 2026, an average of **KES 71,793** per order.
- Nairobi is the strongest branch with **KES 26.1 million (36.4%)** of revenue from **376 orders**, almost three times the smallest branch, Eldoret, at KES 8.8 million (12.3%).
- Laptops earn **KES 41.5 million (57.9%)** of revenue, yet tablets sell the most units (**691 versus 639** for laptops) because they cost less than half as much.
- Monthly revenue peaked in April at **KES 13.52 million** and was lowest in June at **KES 10.90 million**, a gap of **24.0%**, so the half-year ended on a weak note.
- Online orders (**320 orders, KES 22.8 million, 31.7%** of revenue) are worth as much per order as in-store sales (**KES 71,207 versus 70,759**), and Premium online orders in Nairobi and Mombasa alone delivered **83 orders and KES 10.2 million**, which supports investing in the channel.

In [1]:
import numpy as np
import pandas as pd
pd.options.display.float_format = "{:,.1f}".format
pd.set_option("display.width", 200)

## Task 1: First five minutes with the data
The file is produced by `generate_data.py` (seed 2026, unchanged). Dates are parsed on load.

In [2]:
sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
print("shape:", sales.shape)
sales.head()

shape: (1000, 7)


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


In [3]:
sales.info()
print("\nmissing values per column:\n", sales.isna().sum().to_string())
print("\nduplicate order_id:", sales["order_id"].duplicated().sum())
print("date range:", sales["order_date"].min().date(), "to", sales["order_date"].max().date(),
      "|", sales["order_date"].nunique(), "distinct days")

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[us]
 2   branch      1000 non-null   str           
 3   product     1000 non-null   str           
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   str           
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[us](1), int64(3), str(3)
memory usage: 54.8 KB

missing values per column:
 order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0

duplicate order_id: 0
date range: 2026-01-01 to 2026-06-29 | 179 distinct days


In [4]:
for col in ["branch", "product", "channel"]:
    print(sales[col].value_counts().to_string(), "\n")
print(sales["quantity"].describe().to_string())
print(sales.groupby("product")["unit_price"].agg(["min", "max"]).to_string())

branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldoret    117 

product
Laptop        220
Tablet        215
Phone         194
Headphones    187
Charger       184 

channel
In-store       579
Online         320
Phone order    101 

count   1,000.0
mean        3.0
std         1.4
min         1.0
25%         2.0
50%         3.0
75%         4.0
max         5.0
              min    max
product                 
Charger      1200   1200
Headphones   2500   2500
Laptop      65000  65000
Phone       15000  15000
Tablet      28000  28000


**What the inspection shows**
- **Shape and types:** 1,000 orders and 7 columns. `order_date` is a real datetime, `quantity` and `unit_price` are integers, and `branch`, `product` and `channel` are text categories.
- **Quality:** no missing values, no duplicate `order_id`s, and no revenue column yet (it has to be derived).
- **Period:** orders run from 1 January to 29 June 2026 (about six months).
- **Categories:** 5 branches (Nairobi has the most orders), 5 products and 3 channels (In-store, Online, Phone order). `quantity` runs from 1 to 5 and every product has one fixed `unit_price`, so price carries no information beyond product.

## Task 2: Derived columns
`price_band` is based on `unit_price` (Premium at KES 28,000 or more, Value otherwise), so Tablets (28,000) count as Premium.

In [5]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["month"] = sales["order_date"].dt.to_period("M").astype(str)
sales["weekday"] = sales["order_date"].dt.day_name()
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5
sales["price_band"] = np.where(sales["unit_price"] >= 28000, "Premium", "Value")
print("total revenue: KES {:,.0f}".format(sales["revenue"].sum()))
print(sales["price_band"].value_counts().to_string())
sales[["order_date", "revenue", "month", "weekday", "is_weekend", "price_band"]].head()

total revenue: KES 71,792,500
price_band
Value      565
Premium    435


,order_date,revenue,month,weekday,is_weekend,price_band
0,2026-06-03,45000,2026-06,Wednesday,False,Value
1,2026-02-02,260000,2026-02,Monday,False,Premium
2,2026-01-05,140000,2026-01,Monday,False,Premium
3,2026-04-26,12500,2026-04,Sunday,True,Value
4,2026-03-07,7500,2026-03,Saturday,True,Value


## Task 3: Branch ranking by revenue and by orders

In [6]:
branch = sales.groupby("branch").agg(total_revenue=("revenue", "sum"), orders=("order_id", "count"),
                                      avg_order_value=("revenue", "mean"))
branch["share_pct"] = branch["total_revenue"] / branch["total_revenue"].sum() * 100
branch["rank_revenue"] = branch["total_revenue"].rank(ascending=False).astype(int)
branch["rank_orders"] = branch["orders"].rank(ascending=False).astype(int)
branch = branch.sort_values("total_revenue", ascending=False)
branch

,total_revenue,orders,avg_order_value,share_pct,rank_revenue,rank_orders
branch,,,,,,
Nairobi,26106100,376,"69,431.1",36.4,1,1
Mombasa,14003700,186,"75,288.7",19.5,2,2
Nakuru,11970400,162,"73,891.4",16.7,3,3
Kisumu,10868100,159,"68,352.8",15.1,4,4
Eldoret,8844200,117,"75,591.5",12.3,5,5


**Answer: yes, the two rankings are identical** (Nairobi, Mombasa, Nakuru, Kisumu, Eldoret).

Revenue is orders multiplied by average order value. Order counts differ enormously between branches (376 in Nairobi against 117 in Eldoret, 3.2 times as many), while average order values are tightly bunched between KES 68,353 and KES 75,591 (within about 5% of the overall average of KES 71,793). The volume gap swamps the price gap, so whoever takes the most orders also earns the most.

The small differences do show up elsewhere: Eldoret has the **smallest** branch but the **highest** average order (KES 75,591), and Nairobi's average (KES 69,431) is below Mombasa's (KES 75,289). Nairobi leads because of volume, not bigger baskets, so its growth lever is order count while the smaller branches could be helped by more footfall.

## Task 4: Top products by revenue and by units

In [7]:
prod = sales.groupby("product").agg(revenue=("revenue", "sum"), units=("quantity", "sum"), orders=("order_id", "count"))
prod["revenue_share_pct"] = prod["revenue"] / prod["revenue"].sum() * 100
prod["unit_price"] = sales.groupby("product")["unit_price"].first()
print("Top 3 by revenue:\n", prod.nlargest(3, "revenue")[["revenue", "revenue_share_pct"]], "\n")
print("Top 3 by units:\n", prod.nlargest(3, "units")[["units", "unit_price"]])
prod.sort_values("revenue", ascending=False)

Top 3 by revenue:
           revenue  revenue_share_pct
product                             
Laptop   41535000               57.9
Tablet   19348000               26.9
Phone     8880000               12.4 

Top 3 by units:
          units  unit_price
product                   
Tablet     691       28000
Laptop     639       65000
Phone      592       15000


,revenue,units,orders,revenue_share_pct,unit_price
product,,,,,
Laptop,41535000,639,220,57.9,65000
Tablet,19348000,691,215,26.9,28000
Phone,8880000,592,194,12.4,15000
Headphones,1387500,555,187,1.9,2500
Charger,642000,535,184,0.9,1200


**Answer:** both lists contain the same three products (Laptop, Tablet, Phone) but in a different order. By **revenue**: Laptop (KES 41.5 million), Tablet (19.3 million), Phone (8.9 million). By **units**: Tablet (691), Laptop (639), Phone (592).

Laptops and tablets swap places because price, not popularity, drives revenue. Tablets sold 52 more units than laptops, but a laptop costs KES 65,000 against KES 28,000, so laptops earn 2.1 times the revenue and make up **57.9%** of the total. Headphones (555 units) and chargers (535 units) sell almost as many units as phones but add only 1.9% and 0.9% of revenue because they are cheap. Unit volumes are close for all five products, so **price is what separates them**, and laptop availability and pricing deserve the most management attention.

## Task 5: Weekend versus weekday orders

In [8]:
wk = sales.groupby("is_weekend")["revenue"].agg(orders="count", total_revenue="sum", avg_order_revenue="mean")
wk.index = ["Weekday", "Weekend"]
days = pd.date_range(sales["order_date"].min(), sales["order_date"].max())
n_weekend_days = int((days.dayofweek >= 5).sum()); n_weekdays = len(days) - n_weekend_days
wk["calendar_days"] = [n_weekdays, n_weekend_days]
wk["orders_per_day"] = wk["orders"] / wk["calendar_days"]
wk["share_of_orders_pct"] = wk["orders"] / wk["orders"].sum() * 100
wk

,orders,total_revenue,avg_order_revenue,calendar_days,orders_per_day,share_of_orders_pct
Weekday,723,51856200,"71,723.7",128,5.6,72.3
Weekend,277,19936300,"71,972.2",52,5.3,27.7


**Answer:** weekdays had **723 orders and KES 51.9 million** (average KES 71,724 per order); weekends had **277 orders and KES 19.9 million** (average KES 71,972).

Weekends hold 27.7% of orders and 27.8% of revenue, which is slightly below their 28.9% share of calendar days (52 of 180). Per day that is 5.3 orders on weekends against 5.6 on weekdays, about 6% fewer. The average order value is virtually identical (0.3% apart). Customers do not spend more per order at weekends, there are just slightly fewer orders per day, and a gap this small should not drive staffing or promotion decisions.

## Task 6: Best and worst month

In [9]:
monthly = sales.groupby("month")["revenue"].sum().to_frame("revenue")
monthly["days_with_data"] = sales.groupby("month")["order_date"].agg(lambda d: d.dt.days_in_month.iloc[0])
monthly.loc["2026-06", "days_with_data"] = 29   # data stops on 29 June
monthly["revenue_per_day"] = monthly["revenue"] / monthly["days_with_data"]
best, worst = monthly["revenue"].idxmax(), monthly["revenue"].idxmin()
b, w = monthly.loc[best, "revenue"], monthly.loc[worst, "revenue"]
print(f"best: {best} KES {b:,.0f} | worst: {worst} KES {w:,.0f}")
print(f"best is {(b - w) / w * 100:.1f}% higher than worst; worst is {abs(w - b) / b * 100:.1f}% lower than best")
monthly

best: 2026-04 KES 13,518,300 | worst: 2026-06 KES 10,903,400
best is 24.0% higher than worst; worst is 19.3% lower than best


,revenue,days_with_data,revenue_per_day
month,,,
2026-01,11786600,31,"380,212.9"
2026-02,11772000,28,"420,428.6"
2026-03,12396800,31,"399,896.8"
2026-04,13518300,30,"450,610.0"
2026-05,11415400,31,"368,238.7"
2026-06,10903400,29,"375,979.3"


**Answer:** the best month is **April 2026 (KES 13,518,300)** and the worst is **June 2026 (KES 10,903,400)**. April is **24.0% higher** than June, or put the other way, June is 19.3% below April.

April was a clear peak (9.0% above March), after which revenue fell for two months in a row. One caution: months have different lengths and the data ends on 29 June. Per day, May (KES 368k) is actually marginally lower than June (KES 376k), so the weakness is a **May to June slump** and not just one bad month. The cause cannot be seen in this data, so management should check for stock-outs, promotions or seasonality before concluding that demand has weakened.

## Task 7: Premium online orders from Nairobi and Mombasa

In [10]:
mask = (sales["price_band"] == "Premium") & (sales["channel"] == "Online") & sales["branch"].isin(["Nairobi", "Mombasa"])
premium_online = sales[mask]
print("orders:", len(premium_online), "| total revenue: KES {:,.0f}".format(premium_online["revenue"].sum()))
print("share of all revenue: {:.1f}%".format(premium_online["revenue"].sum() / sales["revenue"].sum() * 100))
import os; os.makedirs("outputs", exist_ok=True)
premium_online.to_csv("outputs/premium_online_coast_capital.csv", index=False)
check = pd.read_csv("outputs/premium_online_coast_capital.csv")
print(check.shape, check["price_band"].unique(), check["channel"].unique(), check["branch"].unique())
premium_online.groupby(["branch", "product"])["revenue"].agg(["count", "sum"])

orders: 83 | total revenue: KES 10,218,000
share of all revenue: 14.2%
(83, 12) <StringArray>
['Premium']
Length: 1, dtype: str <StringArray>
['Online']
Length: 1, dtype: str <StringArray>
['Nairobi', 'Mombasa']
Length: 2, dtype: str


count      sum
branch  product                
Mombasa Laptop      10  1820000
        Tablet      23  1876000
Nairobi Laptop      23  4030000
        Tablet      27  2492000

**Answer:** there are **83 orders** worth **KES 10,218,000**, which is 14.2% of all revenue. They are exported to `outputs/premium_online_coast_capital.csv` (no index column). Premium means Tablets and Laptops only, so this slice is purely high-ticket online demand in the capital and the coast.

In [11]:
channel = sales.groupby("channel")["revenue"].agg(orders="count", total_revenue="sum", avg_order_revenue="mean")
channel["share_of_revenue_pct"] = channel["total_revenue"] / channel["total_revenue"].sum() * 100
channel

,orders,total_revenue,avg_order_revenue,share_of_revenue_pct
channel,,,,
In-store,579,40969200,"70,758.5",57.1
Online,320,22786100,"71,206.6",31.7
Phone order,101,8037200,"79,576.2",11.2


**Are online sales worth investing in?** Online is the second-largest channel with 320 orders and 31.7% of revenue. Its average order (KES 71,207) is on par with in-store (KES 70,759), so online customers are not buying smaller baskets. Phone orders have the highest average order (KES 79,576) but only 101 orders. The data supports investing in online, though it says nothing about margins or acquisition costs, which the decision also needs.

## Task 8: Spreadsheet check (`excel/duka_branch_summary.xlsx`)
The workbook has a `Branch_Summary` sheet (SUMIFS revenue, COUNTIFS orders, XLOOKUP region) and a PivotTable of revenue by branch and channel in `Pivot_Branch_Channel`. Here the cached values saved in the workbook are compared with pandas.

In [12]:
xl = pd.read_excel("excel/duka_branch_summary.xlsx", sheet_name="Branch_Summary", header=2, nrows=5)
xl = xl.set_index("Branch")
pandas_rev = sales.groupby("branch")["revenue"].sum()
cmp = pd.DataFrame({"excel_SUMIFS": xl["Total revenue (KES)"], "pandas": pandas_rev}).loc[xl.index]
cmp["match"] = cmp["excel_SUMIFS"] == cmp["pandas"]
print(cmp.to_string())
print("\nregion column from XLOOKUP:", xl["Region"].to_dict())

piv_xl = pd.read_excel("excel/duka_branch_summary.xlsx", sheet_name="Pivot_Branch_Channel", header=5, index_col=0)
piv_pd = sales.pivot_table(index="branch", columns="channel", values="revenue", aggfunc="sum", margins=True, margins_name="Total Result")
piv_xl.columns = piv_xl.columns.str.strip()
piv_pd = piv_pd.rename(columns={"Total": "Total Result"})
piv_xl = piv_xl.loc[piv_pd.index, piv_pd.columns[:3].tolist() + ["Total Result"]]
piv_pd = piv_pd[piv_xl.columns]
print("\nPivotTable equals pandas pivot_table:", bool((piv_xl.values == piv_pd.values).all()))
print("Grand total: Excel pivot {:,.0f} | pandas {:,.0f}".format(piv_xl.loc["Total Result", "Total Result"], sales["revenue"].sum()))
piv_pd

         excel_SUMIFS    pandas  match
Branch                                
Nairobi      26106100  26106100   True
Mombasa      14003700  14003700   True
Nakuru       11970400  11970400   True
Kisumu       10868100  10868100   True
Eldoret       8844200   8844200   True

region column from XLOOKUP: {'Nairobi': 'Capital', 'Mombasa': 'Coast', 'Nakuru': 'Rift Valley', 'Kisumu': 'Lake Region', 'Eldoret': 'Rift Valley'}



PivotTable equals pandas pivot_table: True
Grand total: Excel pivot 71,792,500 | pandas 71,792,500


channel,In-store,Online,Phone order,Total Result
branch,,,,
Eldoret,4885300,3450700,508200,8844200
Kisumu,6468600,3370800,1028700,10868100
Mombasa,8039600,4086800,1877300,14003700
Nairobi,14808400,7527400,3770300,26106100
Nakuru,6767300,4350400,852700,11970400
Total Result,40969200,22786100,8037200,71792500


**Result:** every branch total from the `SUMIFS` table and every cell of the PivotTable equals the pandas result, and both sum to the same grand total of **KES 71,792,500**. The `Region` column uses `XLOOKUP` against a small `Regions` sheet (the region grouping is my own assumption, as the brief gave none). The XLOOKUP values are stored as results, and Excel 365 or Google Sheets will recalculate them when the file is opened.